In [13]:
import math
from dataclasses import dataclass
import google.generativeai as genai
from google.colab import userdata
import textwrap

# 🔐 Always retrieve secrets from Colab Secrets — never hardcode
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)
    print("✅ Gemini API configured successfully.")
except Exception as e:
    print(f"❌ Error: {e}")
    print("Please ensure GOOGLE_API_KEY is set in Colab Secrets.")

client = genai.GenerativeModel('gemini-2.5-flash-lite')


def estimate_tokens(text: str):
    approx_tokens = max(1, math.ceil(len(text) / 4))
    approx_words = len(text.split())
    return {
        "characters": len(text),
        "words": approx_words,
        "estimated_tokens": approx_tokens,
        "rule_of_thumb": "Approximation based on 1 token ~= 4 characters"
    }

def chunk_text(text: str, target_tokens: int = 500, overlap_tokens: int = 50):
    approx_words = max(1, target_tokens)
    overlap_words = max(0, overlap_tokens)
    words = text.split()

    chunks = []
    start = 0
    while start < len(words):
        end = min(len(words), start + approx_words)
        chunk = " ".join(words[start:end]).strip()
        if chunk:
            chunks.append(chunk)
        if end >= len(words):
            break
        start = max(start + 1, end - overlap_words)
    return chunks

@dataclass
class KnowledgeDocument:
    doc_id: str
    title: str
    content: str

KNOWLEDGE_BASE = [
    KnowledgeDocument(
        "doc-01",
        "Azure OpenAI Authentication",
        "Azure OpenAI requires endpoint, deployment name, and API key."
    ),
    KnowledgeDocument(
        "doc-02",
        "Embeddings",
        "Embeddings convert text into vectors for semantic search."
    ),
    KnowledgeDocument(
        "doc-03",
        "Large Context Strategy",
        "Use chunking, retrieval, and summarization for large prompts."
    ),
]

def simple_retrieve(query, docs, top_k=2):
    query_words = set(query.lower().split())
    scored = []

    for doc in docs:
        doc_words = set(doc.content.lower().split())
        score = len(query_words.intersection(doc_words))
        scored.append((score, doc))

    scored.sort(reverse=True, key=lambda x: x[0])
    return [doc for score, doc in scored[:top_k]]

✅ Gemini API configured successfully.


In [14]:
all_text = " ".join([doc.content for doc in KNOWLEDGE_BASE])
response = client.generate_content(f"Answer this question based on the context:\n{all_text}\n\nQuestion: How to handle large prompts?"
  )

print(textwrap.fill(response.text, width=80))
print(response.usage_metadata)

Based on the context provided, to handle large prompts, you should use
**chunking, retrieval, and summarization**.
prompt_token_count: 54
candidates_token_count: 24
total_token_count: 78



In [15]:
query = "How to handle large prompts?"
retrieved_docs = simple_retrieve(query, KNOWLEDGE_BASE, top_k=1)
retrieved_text = " ".join([doc.content for doc in retrieved_docs])
response_opt = client.generate_content(f"Answer this question based on the context:\n{retrieved_text}\n\nQuestion: {query}")

print(textwrap.fill(response_opt.text, width=80))
print(response_opt.usage_metadata)

To handle large prompts, you should use **chunking, retrieval, and
summarization**.
prompt_token_count: 32
candidates_token_count: 18
total_token_count: 50




| Approach | Input Token | Output Token | Total Token | Response Quality | Response Speed |
|----------|----------|----------|----------|----------|----------|
| Baseline  | 54 token | 24 token  | 78 token | Slightly longer  | 2s  |
| Optimized | 32 token  | 18 token  | 50 token  | More Deterministic  | 6s  |


Q : How much token reduction did you achieve?

A : 28 token, 38.8% reduction.
